# Q-TAIRO: Threat-Aware Security Orchestration for SOC Automation
**Notebook implementation scaffold based on the uploaded manuscript.**

This notebook implements an end-to-end *classical, reproducible simulation/prototype* of the manuscript's pipeline using CSE-CIC-IDS2018-style tabular telemetry:
HSTF/schema alignment → deduplication → robust normalization → MI feature selection → SMOTE-Tomek → temporal features → threat severity prediction → asset-aware graph diffusion → alert correlation → response optimization → policy checks and audit log.

**Important research honesty:** the manuscript names quantum components (QFM/VQC/QKL/QAOA/QRL) but does not specify circuit layouts, hyperparameters, equations, labels, or an actual quantum backend. This notebook provides clearly labeled classical proxy implementations so it runs without quantum hardware. Do not report proxy results as quantum-computing results. Replace proxy cells with specified quantum circuits when those details are available.

Upload the dataset in the next cell or set `DATA_PATH`. Expected label column is auto-detected from common CSE-CIC-IDS2018 names; inspect and set it explicitly if needed.


In [ ]:
# 1. Install dependencies (run once in Colab/Jupyter)
%pip -q install pandas numpy scikit-learn imbalanced-learn networkx matplotlib seaborn joblib

In [ ]:
# 2. Imports and reproducibility
import os, re, json, hashlib, warnings, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_auc_score, average_precision_score,
    balanced_accuracy_score, f1_score)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_columns", 100)


In [ ]:
# 3. Load data
# Option A (Colab): upload CSV through the file browser, then set DATA_PATH.
# Option B: mount Drive and set the path.
DATA_PATH = "/content/drive/MyDrive/CSE-CIC-IDS2018.csv"  # <-- CHANGE THIS
if not Path(DATA_PATH).exists():
    print("Dataset path not found. Upload your CSV and update DATA_PATH.")
else:
    df_raw = pd.read_csv(DATA_PATH, low_memory=False)
    print("Loaded:", df_raw.shape)
    display(df_raw.head())


In [ ]:
# 4. HSTF: normalize column names, combine CSVs if needed, detect target
def clean_colname(c):
    c = str(c).strip().replace("\ufeff", "")
    c = re.sub(r"[^A-Za-z0-9]+", "_", c)
    return c.strip("_")

def load_multiple_csv(paths):
    frames = [pd.read_csv(p, low_memory=False) for p in paths]
    return pd.concat(frames, ignore_index=True, sort=False)

# If using multiple telemetry files, replace df_raw with:
# df_raw = load_multiple_csv(["part1.csv", "part2.csv"])

df = df_raw.copy()
df.columns = [clean_colname(c) for c in df.columns]
candidate_targets = ["Label", "label", "Attack", "Class", "target", "Target", "class"]
target_col = next((c for c in candidate_targets if c in df.columns), None)
print("Detected target:", target_col)
if target_col is None:
    print("Columns:", df.columns.tolist())
    raise ValueError("Set target_col manually to the dataset's label column.")


In [ ]:
# 5. Contextual event schema alignment + basic data hygiene
# Map common CIC-IDS column variants to canonical fields where present.
ALIASES = {
    "Flow_Duration": ["Flow_Duration", "FlowDuration"],
    "Source_IP": ["Source_IP", "Src_IP", "SrcIp"],
    "Destination_IP": ["Destination_IP", "Dst_IP", "DstIp"],
    "Source_Port": ["Source_Port", "Src_Port", "SrcPort"],
    "Destination_Port": ["Destination_Port", "Dst_Port", "DstPort"],
    "Protocol": ["Protocol"],
    "Timestamp": ["Timestamp", "Flow_ID", "Date"],
}
rename_map = {}
for canonical, variants in ALIASES.items():
    found = next((v for v in variants if v in df.columns), None)
    if found and found != canonical:
        rename_map[found] = canonical
df = df.rename(columns=rename_map)

# Remove exact duplicate records; normalize target strings.
df = df.drop_duplicates().reset_index(drop=True)
df[target_col] = df[target_col].astype(str).str.strip()
print("After deduplication:", df.shape, "classes:", df[target_col].nunique())
display(df[target_col].value_counts().head(20))


In [ ]:
# 6. MinHash-inspired alert deduplication (deterministic feature-signature proxy)
# Exact duplicate removal above is primary. This optional signature pass collapses near-identical
# rows using a rounded numeric signature. Adjust decimals conservatively for your dataset.
numeric_cols_all = df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols_all:
    sig = pd.util.hash_pandas_object(df[numeric_cols_all].round(3), index=False)
    keep = ~sig.duplicated()
    before = len(df); df = df.loc[keep].reset_index(drop=True)
    print(f"Rounded numeric signature dedup: {before} -> {len(df)} rows")
else:
    print("No numeric columns available for signature dedup.")


In [ ]:
# 7. Build features; drop identifiers / leakage-prone columns
y_text = df[target_col].astype(str)
drop_cols = [target_col]
for c in df.columns:
    if c.lower() in {"flow_id", "source_ip", "destination_ip", "timestamp", "date"}:
        drop_cols.append(c)
X = df.drop(columns=list(set(drop_cols)), errors="ignore").copy()

# Convert categorical predictors; numeric coercion for malformed values.
for c in X.columns:
    if X[c].dtype == "object":
        X[c] = X[c].astype("category").cat.codes.replace(-1, np.nan)
X = X.replace([np.inf, -np.inf], np.nan)
# Remove fully empty and constant features
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1]
print("Feature matrix:", X.shape)


In [ ]:
# 8. Encode target and split BEFORE fitting transforms (avoid leakage)
le = LabelEncoder()
y = le.fit_transform(y_text)
class_names = list(le.classes_)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Classes:", dict(enumerate(class_names)))
print("Train/test:", X_train.shape, X_test.shape)


In [ ]:
# 9. Adaptive Robust Normalization (ARN) + Mutual Information Feature Selection
imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()
Xtr_imp = imputer.fit_transform(X_train)
Xte_imp = imputer.transform(X_test)
Xtr_scaled = scaler.fit_transform(Xtr_imp)
Xte_scaled = scaler.transform(Xte_imp)

mi = mutual_info_classif(Xtr_scaled, y_train, random_state=SEED)
mi_scores = pd.Series(mi, index=X.columns).sort_values(ascending=False)
TOP_K = min(30, X.shape[1])  # manuscript does not specify K; tune on validation data
selected_features = mi_scores.head(TOP_K).index.tolist()
selected_idx = [X.columns.get_loc(c) for c in selected_features]
Xtr_sel = Xtr_scaled[:, selected_idx]
Xte_sel = Xte_scaled[:, selected_idx]
display(mi_scores.head(20).to_frame("MutualInformation"))


In [ ]:
# 10. SMOTE-Tomek (training data only; skip safely for tiny classes)
from collections import Counter
try:
    from imblearn.combine import SMOTETomek
    counts = Counter(y_train)
    min_count = min(counts.values())
    if len(counts) > 1 and min_count >= 2:
        k = min(5, min_count - 1)
        sampler = SMOTETomek(random_state=SEED)
        Xtr_bal, ytr_bal = sampler.fit_resample(Xtr_sel, y_train)
        print("Resampled:", Counter(ytr_bal))
    else:
        Xtr_bal, ytr_bal = Xtr_sel, y_train
        print("Too few samples for SMOTE; using original training set.")
except Exception as e:
    Xtr_bal, ytr_bal = Xtr_sel, y_train
    print("SMOTE-Tomek skipped:", repr(e))


In [ ]:
# 11. Quantum Feature Mapping / VQC / temporal convolution — classical proxy
# The manuscript does not define a circuit or time-window specification. This runnable proxy
# adds nonlinear interactions and rolling statistics when a timestamp/order is available.
def nonlinear_feature_map(A):
    A = np.asarray(A, dtype=np.float32)
    # bounded nonlinear map analogous in spirit to feature embedding, NOT a quantum circuit
    return np.concatenate([A, np.sin(A), np.cos(A), np.tanh(A)], axis=1)

Xtr_embed = nonlinear_feature_map(Xtr_bal)
Xte_embed = nonlinear_feature_map(Xte_sel)
print("Proxy embedding dimensions:", Xtr_embed.shape[1])


In [ ]:
# 12. QPTP: predictive threat-prioritization proxy
# Train multi-class threat classifier. Probabilities are converted into a severity score.
qptp_model = RandomForestClassifier(
    n_estimators=250, max_depth=None, min_samples_leaf=2,
    class_weight="balanced_subsample", random_state=SEED, n_jobs=-1
)
qptp_model.fit(Xtr_embed, ytr_bal)
proba = qptp_model.predict_proba(Xte_embed)
pred = qptp_model.predict(Xte_embed)
confidence = proba.max(axis=1)
# Generic risk score = 0..100 from predicted malicious probability.
benign_idx = next((i for i, name in enumerate(class_names)
                   if any(s in name.lower() for s in ["benign", "normal", "background"])), None)
malicious_prob = 1 - proba[:, benign_idx] if benign_idx is not None else 1 - proba.min(axis=1)
risk_score = np.clip(100 * malicious_prob, 0, 100)
severity = pd.cut(risk_score, [-0.01, 25, 50, 75, 100],
                  labels=["Low", "Moderate", "High", "Critical"]).astype(str)


In [ ]:
# 13. Temporal Hawkes-process-inspired escalation score (proxy)
# Estimate burstiness from ordered test observations. If timestamps exist, sort by time;
# otherwise row order is used. This is a heuristic, not fitted Hawkes-process inference.
def temporal_escalation(scores, window=20):
    s = pd.Series(scores)
    rolling = s.rolling(window=min(window, max(2, len(s))), min_periods=1).mean()
    denom = max(float(rolling.max()), 1e-8)
    return np.clip(rolling.to_numpy() / denom, 0, 1)

temporal_escalation_prob = temporal_escalation(risk_score)


In [ ]:
# 14. Asset-aware graph risk diffusion (AGRD proxy)
# Construct an asset graph from available source/destination IP fields. If unavailable,
# create a feature-similarity graph over alerts. Diffuse risk with normalized adjacency.
def graph_diffuse_risk(scores, k=5, alpha=0.35):
    n = len(scores)
    if n <= 1: return np.asarray(scores, dtype=float)
    # k-nearest-neighbor graph on selected feature vectors
    from sklearn.neighbors import kneighbors_graph
    kk = min(k, n-1)
    A = kneighbors_graph(Xte_sel, n_neighbors=kk, mode="connectivity", include_self=False)
    A = A.maximum(A.T).astype(float)
    deg = np.asarray(A.sum(axis=1)).ravel()
    deg[deg == 0] = 1
    P = A.multiply(1 / deg[:, None])
    r = np.asarray(scores, dtype=float) / 100
    for _ in range(8):
        r = (1-alpha) * (np.asarray(scores, dtype=float)/100) + alpha * P.dot(r)
    return np.clip(r * 100, 0, 100)

asset_graph_risk = graph_diffuse_risk(risk_score)


In [ ]:
# 15. Alert correlation: graph communities + incident/attack-path approximation
G = nx.Graph()
for i, score in enumerate(asset_graph_risk):
    G.add_node(i, risk=float(score), predicted_class=class_names[int(pred[i])])
# Reuse similarity graph edges
try:
    from sklearn.neighbors import kneighbors_graph
    A = kneighbors_graph(Xte_sel, n_neighbors=min(5, max(1, len(Xte_sel)-1)),
                         mode="connectivity", include_self=False).maximum(
                         kneighbors_graph(Xte_sel, n_neighbors=min(5, max(1, len(Xte_sel)-1)),
                         mode="connectivity", include_self=False).T)
    coo = A.tocoo()
    for u, v in zip(coo.row, coo.col):
        if u < v: G.add_edge(int(u), int(v))
except Exception as e:
    print("Graph construction warning:", e)

communities = list(nx.community.greedy_modularity_communities(G)) if G.number_of_edges() else [{i} for i in G.nodes]
community_id = {}
for cid, members in enumerate(communities):
    for node in members: community_id[node] = cid
print("Correlated alert communities:", len(communities))


In [ ]:
# 16. QARO: response action optimization (risk/disruption/containment proxy)
# Candidate actions and illustrative utility coefficients. Calibrate using SOC policy/incident data.
ACTIONS = {
    "Monitor":       {"containment": 0.10, "disruption": 0.00, "cost": 0.05},
    "Rate-limit":    {"containment": 0.45, "disruption": 0.15, "cost": 0.10},
    "Block-IP":      {"containment": 0.75, "disruption": 0.30, "cost": 0.15},
    "Isolate-host":  {"containment": 0.90, "disruption": 0.65, "cost": 0.25},
    "Disable-account":{"containment": 0.85, "disruption": 0.50, "cost": 0.20},
}
MAX_DISRUPTION = 0.70
RESIDUAL_RISK_LIMIT = 35.0

def choose_response(risk):
    candidates = []
    for name, p in ACTIONS.items():
        residual = risk * (1 - p["containment"])
        disruption = p["disruption"]
        utility = (risk - residual) - 25*disruption - 10*p["cost"]
        feasible = disruption <= MAX_DISRUPTION and residual <= RESIDUAL_RISK_LIMIT
        candidates.append((feasible, utility, name, residual, disruption))
    feasible = [x for x in candidates if x[0]]
    best = max(feasible or candidates, key=lambda x: x[1])
    return best[2], best[3], best[4]

response = [choose_response(float(r)) for r in asset_graph_risk]
response_action = [r[0] for r in response]
residual_risk = [r[1] for r in response]
disruption = [r[2] for r in response]


In [ ]:
# 17. Policy-as-code, SOAR playbook simulation, and append-only audit chain
POLICY = {
    "auto_execute_max_risk": 80,
    "approved_actions": set(ACTIONS),
    "require_human_approval": {"Isolate-host", "Disable-account"},
}
def policy_decision(risk, action):
    if action not in POLICY["approved_actions"]:
        return "DENY"
    if action in POLICY["require_human_approval"]:
        return "REQUIRES_APPROVAL"
    if risk >= POLICY["auto_execute_max_risk"]:
        return "REQUIRES_APPROVAL"
    return "ALLOW_SIMULATED"

def append_audit(records):
    prev = "GENESIS"
    output = []
    for rec in records:
        payload = json.dumps(rec, sort_keys=True, default=str)
        digest = hashlib.sha256((prev + payload).encode()).hexdigest()
        output.append({**rec, "previous_hash": prev, "record_hash": digest})
        prev = digest
    return output

policy_status = [policy_decision(float(r), a) for r, a in zip(asset_graph_risk, response_action)]
audit_records = append_audit([
    {"alert_id": int(i), "predicted_label": class_names[int(pred[i])],
     "risk": round(float(asset_graph_risk[i]), 3), "severity": str(severity[i]),
     "community": int(community_id.get(i, -1)), "action": response_action[i],
     "residual_risk": round(float(residual_risk[i]), 3), "policy": policy_status[i]}
    for i in range(len(pred))
])
results = pd.DataFrame(audit_records)
display(results.head())


In [ ]:
# 18. Evaluation
print("Accuracy:", accuracy_score(y_test, pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=class_names, zero_division=0))
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(max(8, len(class_names)*0.7), max(6, len(class_names)*0.6)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("QPTP Proxy — Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.tight_layout(); plt.show()

# One-vs-rest ROC-AUC / PR-AUC when valid
try:
    print("ROC-AUC (OVR macro):", roc_auc_score(y_test, proba, multi_class="ovr", average="macro"))
except Exception as e: print("ROC-AUC unavailable:", e)
try:
    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(y_test, classes=np.arange(len(class_names)))
    print("PR-AUC (macro OVR):", average_precision_score(y_bin, proba, average="macro"))
except Exception as e: print("PR-AUC unavailable:", e)


In [ ]:
# 19. SOC dashboards: risk distribution and response action counts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(asset_graph_risk, bins=20, ax=axes[0])
axes[0].set_title("Asset-aware propagated risk")
axes[0].set_xlabel("Risk score (0–100)")
results["response_action"].value_counts().plot(kind="bar", ax=axes[1])
axes[1].set_title("Recommended response actions")
axes[1].set_ylabel("Alert count")
plt.tight_layout(); plt.show()

display(results.groupby(["severity", "response_action"]).size().unstack(fill_value=0))


In [ ]:
# 20. Export artifacts
OUT_DIR = Path("q_tairo_outputs")
OUT_DIR.mkdir(exist_ok=True)
results.to_csv(OUT_DIR / "Q_TAIRO_alert_response_audit.csv", index=False)
mi_scores.to_csv(OUT_DIR / "feature_mutual_information.csv", header=["MI_score"])
pd.DataFrame({"feature": selected_features}).to_csv(OUT_DIR / "selected_features.csv", index=False)
import joblib
joblib.dump({
    "imputer": imputer, "scaler": scaler, "label_encoder": le,
    "selected_features": selected_features, "model": qptp_model,
    "actions": ACTIONS, "policy": POLICY
}, OUT_DIR / "q_tairo_proxy_model.joblib")
print("Saved artifacts to:", OUT_DIR.resolve())


## Interpretation and limitations
- **QPTP prediction** is a supervised threat-classification proxy. For true temporal forecasting, create chronological windows and labels such as future escalation within a defined horizon.
- **Hawkes process** is represented by a rolling risk-burst score; fit a point-process model when event timestamps and event types are available.
- **AGRD** uses feature-neighbor diffusion because asset identity/topology may not be present in the CSV. Replace with a real asset graph from CMDB, EDR, identity, and network telemetry.
- **Alert correlation** uses similarity graph communities, not a full heterogeneous graph attention network or Leiden implementation.
- **QARO** is a transparent heuristic constrained optimizer, not QAOA/CVaR/QRL. Define operational cost, disruption, containment, risk tolerance, and action preconditions from SOC policy.
- **Blockchain audit** is represented by a SHA-256 hash chain. This is tamper-evident in sequence but not a distributed blockchain.
- Use only in an isolated test environment; automated actions here are simulated and do not connect to firewalls, EDR, IAM, or production SOAR.
